# Download and load the Qwen2.5-0.5B-Instruct model and tokenizer

In [90]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Hugging Face model identifier
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# Download and load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Download and load the model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print(model.device)  # Check the device of the model
print(model.training)  # Check if the model is in training mode

print("Model and tokenizer loaded successfully!")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

cuda:0
False
Model and tokenizer loaded successfully!


# Define the Chatbot / Implement RAG

In [91]:
import re
import torch
import pandas as pd
from sentence_transformers import SentenceTransformer

SYSTEM_PROMPT = (
    "You are Bob, a friendly movie expert. Answer ONLY using the movie "
    "information provided. If it doesn't contain the answer, say you don't know. "
    "Reply in 3-4 sentences."
)

def norm(s):
    return " ".join(re.sub(r"[^a-z0-9 ]", " ", s.lower()).split())

class BobTheChatbot:
    def __init__(self, model, tokenizer, max_history_turns=3):
        self.model, self.tokenizer = model, tokenizer
        self.history = []
        self.max_history_turns = max_history_turns

        df = pd.read_csv('wiki_movie_plots_deduped.csv',
                         usecols=['Release Year', 'Title', 'Plot'])
        self.norm_titles = [f" {norm(t)} " for t in df["Title"]]
        self.docs = (df["Title"] + " (" + df["Release Year"].astype(str) + "): " + df["Plot"]).tolist()

        self.sentence_model = SentenceTransformer('all-MiniLM-L6-v2')
        self.embeddings = self.sentence_model.encode(
            self.docs, convert_to_tensor=True, batch_size=128, show_progress_bar=True
        )

    def retrieve(self, prompt):
        # 1) exact title match (longest title wins, e.g. "The Dark Knight Rises" > "The Dark Knight")
        p = f" {norm(prompt)} "
        hits = [i for i, t in enumerate(self.norm_titles) if len(t) > 6 and t in p]
        if hits:
            return self.docs[max(hits, key=lambda i: len(self.norm_titles[i]))]
        # 2) semantic fallback
        q = self.sentence_model.encode(prompt, convert_to_tensor=True)
        scores = torch.nn.functional.cosine_similarity(q, self.embeddings)
        return self.docs[scores.argmax().item()]

    def chatWithRAG(self, prompt, max_new_tokens=200):
        doc = self.retrieve(prompt)[:2000]  # keep the plot a sane length
        messages = (
            [{"role": "system", "content": SYSTEM_PROMPT}]
            + self.history
            + [{"role": "user", "content": f"Movie information:\n{doc}\n\nQuestion: {prompt}"}]
        )
        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)
        out = self.model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            repetition_penalty=1.05, pad_token_id=self.tokenizer.eos_token_id,
        )
        answer = self.tokenizer.decode(
            out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
        ).strip()

        # history keeps only the bare Q&A, not the retrieved plot
        self.history += [{"role": "user", "content": prompt},
                         {"role": "assistant", "content": answer}]
        self.history = self.history[-2 * self.max_history_turns:]
        return answer

# Create the Chatbot

In [92]:
bob = BobTheChatbot(model, tokenizer)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/273 [00:00<?, ?it/s]

# Call Chat

In [93]:
bob.chatWithRAG("Hello Bob! Can you tell me something about the movie Handphone?")

'The movie "Handphone" (2009) is a romantic comedy-drama that follows talent manager Seung-min as he struggles to turn his life around after losing his job due to a sexual scandal. Seung-min meets rising actress Yoon Jin-ah, who becomes his one last hope for redemption. However, Seung-min\'s pursuit of Yoon leads him into a dangerous situation when he receives a threatening message from her ex-lover. Seung-min must navigate this crisis while trying to retrieve the sex tape that could prove his innocence. The film explores themes of love, redemption, and the consequences of pursuing one\'s dreams despite personal challenges.'

In [94]:
bob.chatWithRAG("Can you tell me about the movie The Dark Knight?")

'The Dark Knight is a 2008 superhero film directed by Christopher Nolan. It tells the story of Batman and his allies, including Batman himself, who are tasked with stopping a group of criminals who have taken over Gotham City. The film features a mix of action sequences, suspenseful plot twists, and memorable characters such as Batman, Rachel Dawes, and Detective Gordon. The movie explores themes of justice, redemption, and the power of teamwork in the face of adversity.'

In [95]:
bob.chatWithRAG("Can you tell me something about the movie Swades?")

'Yes, I can tell you about the movie "Swades" (2004). It is a Bollywood film directed by Shah Rukh Khan. The movie stars Shah Rukh Khan as Mohan Bhargava, a young Indian who works as a Project Manager on the Global Precipitation Measurement (GPM) program at NASA in the United States. Mohan is a character who is struggling to find his place in the world and is facing various challenges in his life. The film is set against the backdrop of India and focuses on the journey of Mohan as he tries to make a change in his life.'

In [96]:
bob.chatWithRAG("Who played the lead role in the movie Swades?")

'In the movie "Swades" (2004), the lead role was played by Shah Rukh Khan.'